# 面试问题：怎样从零实现 U-Net，并处理 skip connection 的奇数尺寸对齐？

## 可直接复述的回答主线

1. U-Net 编码器通过卷积和池化提取低分辨率语义，解码器逐级上采样恢复像素位置。
2. 同尺度 skip connection 把编码器的高分辨率边界特征与解码器语义特征在通道维拼接。
3. 奇数高宽经过向下取整池化后无法被转置卷积精确还原，因此 concat 前必须显式裁剪或 padding 对齐。
4. 二值分割头输出每像素一个 logit，训练用 BCEWithLogits 与 soft Dice，sigmoid 只在概率和阈值评测时使用。
5. 评测应给出同数据阈值基线、逐图 IoU/Dice、输入与 mask 字符图、skip shape、像素概率和梯度。
6. 空 mask、阈值选择、resize 规则和后处理必须写入评估合同，否则线上与离线指标不可比。
7. 生产还需真实标注切分、边界噪声、类别长尾、大图 tile、校准、导出对齐和像素漂移监控。

下面用同一批可读输入依次验证朴素基线、手写核心机制、中间过程、失败修正与生产边界。

## 1. 真实案例与输入预览

案例是 8 张 17×19 脱敏包装质检灰度图，前景为圆形污渍或矩形破损，包含不同位置、大小和亮度。奇数尺寸会真实触发两级 pooling 后的 skip 对齐问题；内置图只用于解释机制，不代表医学或工业真实分割精度。

In [1]:
import math  # 计算训练梯度和分割指标。
import warnings  # 过滤本地 PyTorch 环境的无关兼容警告。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 保持输出聚焦分割实验。
import torch  # 使用基础卷积与张量运算手写 U-Net。
torch.manual_seed(231)  # 固定模型初始化与训练轨迹。
torch.set_num_threads(1)  # 固定 CPU 单线程提高可复现性。
height = 17  # 定义会触发奇数对齐的图像高度。
width = 19  # 定义会触发奇数对齐的图像宽度。
yy, xx = torch.meshgrid(torch.arange(height), torch.arange(width), indexing="ij")  # 构造像素坐标网格。
images = []  # 保存八张质检灰度图。
masks = []  # 保存八张像素级真值 mask。
sample_ids = []  # 保存可读样本编号。
shape_names = []  # 保存每张图的缺陷形状说明。
for index in range(8):  # 生成至少六张具有不同位置和亮度的样本。
    if index % 2 == 0:  # 偶数样本构造圆形污渍。
        center_y = 5 + (index * 3) % 8  # 计算当前圆心纵坐标。
        center_x = 5 + (index * 4) % 10  # 计算当前圆心水平坐标。
        radius = 2 + index % 3  # 在二到四像素半径间变化。
        mask_image = (((yy - center_y) ** 2 + (xx - center_x) ** 2) <= radius ** 2).to(torch.float32)  # 生成圆形精确真值。
        shape_name = f"圆形r={radius}"  # 保存当前形状描述。
    else:  # 奇数样本构造矩形破损。
        top = 2 + index % 5  # 计算矩形顶部位置。
        left = 3 + (index * 2) % 8  # 计算矩形左侧位置。
        rect_height = 3 + index % 3  # 设置三到五像素高度。
        rect_width = 4 + index % 4  # 设置四到七像素宽度。
        mask_image = ((yy >= top) & (yy < top + rect_height) & (xx >= left) & (xx < left + rect_width)).to(torch.float32)  # 生成矩形精确真值。
        shape_name = f"矩形{rect_height}x{rect_width}"  # 保存当前形状描述。
    foreground_level = 0.48 + 0.06 * (index % 4)  # 让部分前景低于固定 0.65 基线阈值。
    background = 0.10 + 0.05 * xx.to(torch.float32) / (width - 1) + 0.02 * torch.sin((yy + index) * 0.8)  # 构造有横向渐变的包装背景。
    image = background + foreground_level * mask_image  # 把不同亮度缺陷叠加到背景。
    images.append(image.unsqueeze(0))  # 保存单通道输入图。
    masks.append(mask_image.unsqueeze(0))  # 保存单通道二值 mask。
    sample_ids.append(f"pack-{index + 1:02d}")  # 生成稳定样本 ID。
    shape_names.append(shape_name)  # 保存当前缺陷形状。
images = torch.stack(images)  # 堆叠为八乘一乘十七乘十九张量。
masks = torch.stack(masks)  # 堆叠对应像素真值。
def render_mask(mask_image):  # 把二值 mask 转为可读字符图。
    return "\n".join("".join("#" if float(pixel) >= 0.5 else "." for pixel in row) for row in mask_image.squeeze(0))  # 用井号显示前景区域。
print("教学实验输入：包装缺陷分割，image/mask shapes=", tuple(images.shape), tuple(masks.shape))  # 展示图像批量和奇数尺寸。
for index in range(len(images)):  # 逐样本展示缺陷字段与真值形状。
    foreground_pixels = int(masks[index].sum().item())  # 统计当前前景像素数。
    mean_foreground = float(images[index][masks[index].bool()].mean().item())  # 计算当前缺陷平均亮度。
    print(f"\n{sample_ids[index]} {shape_names[index]} foreground_pixels={foreground_pixels} mean_foreground={mean_foreground:.3f}\n{render_mask(masks[index])}")  # 输出真实 mask 字符图与可读统计。

教学实验输入：包装缺陷分割，image/mask shapes= (8, 1, 17, 19) (8, 1, 17, 19)

pack-01 圆形r=2 foreground_pixels=13 mean_foreground=0.583
...................
...................
...................
.....#.............
....###............
...#####...........
....###............
.....#.............
...................
...................
...................
...................
...................
...................
...................
...................
...................

pack-02 矩形4x5 foreground_pixels=20 mean_foreground=0.647
...................
...................
...................
.....#####.........
.....#####.........
.....#####.........
.....#####.........
...................
...................
...................
...................
...................
...................
...................
...................
...................
...................

pack-03 圆形r=4 foreground_pixels=49 mean_foreground=0.733
...................
...................
...................
...................
...

## 2. Baseline / 基线：固定像素阈值 0.65

直接把亮度不低于 0.65 的像素判为缺陷，不需要训练，但会漏掉暗污渍，还可能把背景高亮位置误判。基线和 U-Net 使用完全相同的八张输入与 IoU 公式。

In [2]:
def per_image_iou(predictions, targets):  # 计算批次中每张图的硬 IoU。
    predicted = predictions.to(torch.bool)  # 把预测转换为布尔前景。
    truth = targets.to(torch.bool)  # 把真值转换为布尔前景。
    intersection = (predicted & truth).flatten(1).sum(dim=1).to(torch.float32)  # 统计逐图交集像素。
    union = (predicted | truth).flatten(1).sum(dim=1).to(torch.float32)  # 统计逐图并集像素。
    return torch.where(union > 0, intersection / union, torch.ones_like(union))  # 对双空样本按一分处理。
baseline_masks = images >= 0.65  # 用固定亮度阈值生成基线像素预测。
baseline_ious = per_image_iou(baseline_masks, masks)  # 计算八张图各自基线 IoU。
baseline_mean_iou = float(baseline_ious.mean().item())  # 汇总固定阈值平均 IoU。
print("Baseline：pixel >= 0.65")  # 标记下表为无训练阈值法。
print("sample   shape       foreground_mean  true_pixels  predicted_pixels  IoU")  # 输出逐图基线表头。
for index in range(len(images)):  # 逐图展示阈值法漏检程度。
    foreground_mean = float(images[index][masks[index].bool()].mean().item())  # 读取当前前景平均亮度。
    print(f"{sample_ids[index]:<8} {shape_names[index]:<11} {foreground_mean:>15.3f} {int(masks[index].sum()):>12} {int(baseline_masks[index].sum()):>16} {baseline_ious[index].item():.4f}")  # 输出像素数与 IoU。
print(f"Baseline mean IoU={baseline_mean_iou:.4f}")  # 展示模型需要超越的同数据指标。

Baseline：pixel >= 0.65
sample   shape       foreground_mean  true_pixels  predicted_pixels  IoU
pack-01  圆形r=2                 0.583           13                0 0.0000
pack-02  矩形4x5                 0.647           20                6 0.3000
pack-03  圆形r=4                 0.733           49               49 1.0000
pack-04  矩形3x7                 0.806           21               21 1.0000
pack-05  圆形r=3                 0.604           29                0 0.0000
pack-06  矩形5x5                 0.667           25               21 0.8400
pack-07  圆形r=2                 0.713           13               13 1.0000
pack-08  矩形4x7                 0.786           28               28 1.0000
Baseline mean IoU=0.6425


## 3. 底层实现：DoubleConv、Down、Up、显式对齐和像素损失

两级编码器把 17×19 依次变成 8×9 和 4×4；转置卷积只能恢复 8×8、16×18，因此 `align_to` 在每次 concat 前补齐到 skip 尺寸。输出保持 logits，损失由 BCE 与 soft Dice 组成。

In [3]:
def align_to(source, reference):  # 把 decoder 特征中心裁剪或对称 padding 到 skip 尺寸。
    target_height, target_width = reference.shape[-2:]  # 读取目标 skip 空间尺寸。
    source_height, source_width = source.shape[-2:]  # 读取上采样后 decoder 尺寸。
    if source_height > target_height:  # 检查高度是否需要中心裁剪。
        top = (source_height - target_height) // 2  # 计算顶部裁剪起点。
        source = source[..., top:top + target_height, :]  # 裁剪多余高度。
    if source_width > target_width:  # 检查宽度是否需要中心裁剪。
        left = (source_width - target_width) // 2  # 计算左侧裁剪起点。
        source = source[..., :, left:left + target_width]  # 裁剪多余宽度。
    height_gap = target_height - source.shape[-2]  # 计算仍需补齐的高度。
    width_gap = target_width - source.shape[-1]  # 计算仍需补齐的宽度。
    return torch.nn.functional.pad(source, [width_gap // 2, width_gap - width_gap // 2, height_gap // 2, height_gap - height_gap // 2])  # 按左右和上下尽量对称补零。
class DoubleConv(torch.nn.Module):  # 定义保持空间尺寸的双卷积模块。
    def __init__(self, input_channels, output_channels):  # 初始化两层三乘三卷积。
        super().__init__()  # 注册卷积参数。
        self.network = torch.nn.Sequential(torch.nn.Conv2d(input_channels, output_channels, 3, padding=1), torch.nn.ReLU(), torch.nn.Conv2d(output_channels, output_channels, 3, padding=1), torch.nn.ReLU())  # 串联两次局部特征提取。
    def forward(self, inputs):  # 对输入执行两次卷积与激活。
        return self.network(inputs)  # 返回空间尺寸不变的特征。
class Down(torch.nn.Module):  # 定义一次池化加双卷积编码级。
    def __init__(self, input_channels, output_channels):  # 初始化下采样模块。
        super().__init__()  # 注册池化与卷积子模块。
        self.network = torch.nn.Sequential(torch.nn.MaxPool2d(2), DoubleConv(input_channels, output_channels))  # 先向下取整池化再提取语义。
    def forward(self, inputs):  # 执行编码器下采样。
        return self.network(inputs)  # 返回一半空间分辨率特征。
class Up(torch.nn.Module):  # 定义转置卷积、对齐、拼接和融合模块。
    def __init__(self, decoder_channels, skip_channels, output_channels):  # 初始化上采样和 concat 后卷积。
        super().__init__()  # 注册解码器参数。
        self.up = torch.nn.ConvTranspose2d(decoder_channels, output_channels, 2, stride=2)  # 将 decoder 空间尺寸精确翻倍。
        self.fuse = DoubleConv(output_channels + skip_channels, output_channels)  # 融合上采样和高分辨率 skip。
    def forward(self, decoder_feature, skip_feature, return_debug=False):  # 执行单级解码并按需返回 shape。
        raw_upsampled = self.up(decoder_feature)  # 生成尚未对齐的翻倍特征。
        aligned = align_to(raw_upsampled, skip_feature)  # 裁剪或补零到 skip 空间尺寸。
        merged = torch.cat([skip_feature, aligned], dim=1)  # 沿通道维拼接定位和语义信息。
        output = self.fuse(merged)  # 用双卷积融合两路特征。
        debug = {"raw_upsampled": raw_upsampled, "aligned": aligned, "merged": merged}  # 保存对齐前后和拼接张量。
        return (output, debug) if return_debug else output  # 按需返回对齐证据。
class UNet(torch.nn.Module):  # 定义两级编码和两级解码的完整 U-Net。
    def __init__(self, base_channels=8):  # 初始化五个特征模块与像素头。
        super().__init__()  # 注册完整分割网络参数。
        self.input_conv = DoubleConv(1, base_channels)  # 提取原分辨率边界特征。
        self.down_one = Down(base_channels, base_channels * 2)  # 下采样到八乘九。
        self.down_two = Down(base_channels * 2, base_channels * 4)  # 下采样到四乘四 bottleneck。
        self.up_one = Up(base_channels * 4, base_channels * 2, base_channels * 2)  # 恢复并融合第一层 skip。
        self.up_two = Up(base_channels * 2, base_channels, base_channels)  # 恢复并融合原分辨率 skip。
        self.output_conv = torch.nn.Conv2d(base_channels, 1, 1)  # 为每个像素输出一个二分类 logit。
    def forward(self, inputs, return_debug=False):  # 执行完整编码器—解码器前向。
        skip_one = self.input_conv(inputs)  # 生成十七乘十九高分辨率 skip。
        skip_two = self.down_one(skip_one)  # 生成八乘九中分辨率 skip。
        bottleneck = self.down_two(skip_two)  # 生成四乘四语义 bottleneck。
        decoded_one, first_debug = self.up_one(bottleneck, skip_two, return_debug=True)  # 第一次上采样并对齐八乘九。
        decoded_two, second_debug = self.up_two(decoded_one, skip_one, return_debug=True)  # 第二次上采样并对齐十七乘十九。
        logits = self.output_conv(decoded_two)  # 生成与输入等大的像素 logits。
        debug = {"skip_one": skip_one, "skip_two": skip_two, "bottleneck": bottleneck, "decoded_one": decoded_one, "decoded_two": decoded_two, "first_up": first_debug, "second_up": second_debug}  # 汇总 U 形路径中间张量。
        return (logits, debug) if return_debug else logits  # 按需返回解释张量。
def soft_dice_loss(logits, targets, epsilon=1.0e-6):  # 计算可微分区域重叠损失。
    probabilities = torch.sigmoid(logits)  # 把像素 logits 转为前景概率。
    intersection = (probabilities * targets).flatten(1).sum(dim=1)  # 计算逐图软交集。
    denominator = probabilities.flatten(1).sum(dim=1) + targets.flatten(1).sum(dim=1)  # 计算逐图预测和真值面积和。
    dice = (2.0 * intersection + epsilon) / (denominator + epsilon)  # 计算数值稳定 soft Dice。
    return 1.0 - dice.mean()  # 返回批次平均 Dice loss。
model = UNet()  # 创建待训练的手写 U-Net。
optimizer = torch.optim.Adam(model.parameters(), lr=0.015)  # 创建像素分割优化器。
history = []  # 保存真实 backward 的损失和梯度轨迹。
for step in range(260):  # 对八张图执行受控全批次训练。
    optimizer.zero_grad(set_to_none=True)  # 清除上一步全部参数梯度。
    logits, training_debug = model(images, return_debug=True)  # 前向执行 U 形编码、对齐和解码。
    bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, masks)  # 计算逐像素稳定二元交叉熵。
    dice_loss = soft_dice_loss(logits, masks)  # 计算区域重叠损失。
    loss = 0.5 * bce + 0.5 * dice_loss  # 等权合并像素概率和区域目标。
    loss.backward()  # 对所有卷积和转置卷积执行真实反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters() if parameter.grad is not None))  # 汇总全部非空梯度二范数。
    optimizer.step()  # 应用 Adam 更新 U-Net 参数。
    if step % 65 == 0 or step == 259:  # 每六十五步保存训练证据。
        history.append({"step": step, "loss": loss.item(), "bce": bce.item(), "dice_loss": dice_loss.item(), "gradient_norm": gradient_norm})  # 保存损失分项与梯度。
model.eval()  # 切换到确定性推理模式。
with torch.no_grad():  # 取得最终像素概率和中间 shape。
    final_logits, final_debug = model(images, return_debug=True)  # 对八张图执行最终分割前向。
    final_probabilities = torch.sigmoid(final_logits)  # 把最终 logits 转为前景概率。
print("U-Net训练轨迹=", history)  # 展示 BCE、Dice 和梯度变化。
print("U-Net shape trace=", {name: tuple(value.shape) for name, value in final_debug.items() if isinstance(value, torch.Tensor)})  # 展示编码器和解码器各尺度。
print("第二次上采样 raw/aligned/merged=", tuple(final_debug["second_up"]["raw_upsampled"].shape), tuple(final_debug["second_up"]["aligned"].shape), tuple(final_debug["second_up"]["merged"].shape))  # 展示 16×18 到 17×19 的真实对齐过程。
print("pack-01中心5x5概率=", torch.round(final_probabilities[0, 0, 4:9, 4:9] * 1000) / 1000)  # 展示局部逐像素概率而非只看最终 mask。

U-Net训练轨迹= [{'step': 0, 'loss': 0.8047101497650146, 'bce': 0.7408666014671326, 'dice_loss': 0.8685537576675415, 'gradient_norm': 0.27800438907897357}, {'step': 65, 'loss': 0.06739512085914612, 'bce': 0.04196661710739136, 'dice_loss': 0.09282362461090088, 'gradient_norm': 1.3103726282233614}, {'step': 130, 'loss': 0.00047575830831192434, 'bce': 0.00017421245865989476, 'dice_loss': 0.0007773041725158691, 'gradient_norm': 0.014561053462758482}, {'step': 195, 'loss': 6.0365855460986495e-05, 'bce': 1.982104549824726e-05, 'dice_loss': 0.00010091066360473633, 'gradient_norm': 0.0006794432081180072}, {'step': 259, 'loss': 2.760838469839655e-05, 'bce': 8.665540008223616e-06, 'dice_loss': 4.655122756958008e-05, 'gradient_norm': 0.00038095741388434197}]
U-Net shape trace= {'skip_one': (8, 8, 17, 19), 'skip_two': (8, 16, 8, 9), 'bottleneck': (8, 32, 4, 4), 'decoded_one': (8, 16, 8, 9), 'decoded_two': (8, 8, 17, 19)}
第二次上采样 raw/aligned/merged= (8, 8, 16, 18) (8, 8, 17, 19) (8, 16, 17, 19)
pack-01中心

## 4. 逐图分割结果与结果解读

使用固定 0.5 阈值把 U-Net 概率转为 mask，在同一八张图上比较固定亮度基线与模型 IoU，并显示四个原始真值/预测字符图。

In [4]:
predicted_masks = final_probabilities >= 0.5  # 使用预先声明的 0.5 概率阈值生成硬 mask。
model_ious = per_image_iou(predicted_masks, masks)  # 计算 U-Net 每张图的 IoU。
model_mean_iou = float(model_ious.mean().item())  # 汇总八张图平均 IoU。
print("sample   shape       baseline_IoU  U-Net_IoU  predicted_pixels  true_pixels")  # 输出逐图同数据结果表头。
for index in range(len(images)):  # 逐图比较阈值基线和 U-Net。
    print(f"{sample_ids[index]:<8} {shape_names[index]:<11} {baseline_ious[index].item():>12.4f} {model_ious[index].item():>10.4f} {int(predicted_masks[index].sum()):>16} {int(masks[index].sum()):>11}")  # 输出当前形状、两种 IoU 和区域大小。
for index in [0, 1, 4, 7]:  # 选择圆形、矩形、暗目标和亮目标展示像素结果。
    print(f"\n{sample_ids[index]} gold\n{render_mask(masks[index])}\nprediction\n{render_mask(predicted_masks[index])}")  # 输出逐像素真值与 U-Net 预测字符图。
print(f"结果解读：固定亮度 baseline mean IoU={baseline_mean_iou:.4f}，手写U-Net={model_mean_iou:.4f}；模型学会同时利用局部边界和上下文。")  # 解释同数据分割收益与受控范围。

sample   shape       baseline_IoU  U-Net_IoU  predicted_pixels  true_pixels
pack-01  圆形r=2             0.0000     1.0000               13          13
pack-02  矩形4x5             0.3000     1.0000               20          20
pack-03  圆形r=4             1.0000     1.0000               49          49
pack-04  矩形3x7             1.0000     1.0000               21          21
pack-05  圆形r=3             0.0000     1.0000               29          29
pack-06  矩形5x5             0.8400     1.0000               25          25
pack-07  圆形r=2             1.0000     1.0000               13          13
pack-08  矩形4x7             1.0000     1.0000               28          28

pack-01 gold
...................
...................
...................
.....#.............
....###............
...#####...........
....###............
.....#.............
...................
...................
...................
...................
...................
...................
...................
..................

## 5. 失败案例与修正：直接拼接 16×18 decoder 与 17×19 skip

第二级转置卷积只能把 8×9 的输入变成 16×18，而原始 skip 是 17×19。直接 `cat` 会抛出尺寸错误；先用明确的对称 padding 对齐后即可拼接。

In [5]:
raw_second_up = model.up_two.up(final_debug["decoded_one"])  # 取得未经过 align_to 的十六乘十八特征。
target_skip = final_debug["skip_one"]  # 取得十七乘十九高分辨率 skip。
try:  # 尝试复现错误的直接 concat。
    torch.cat([target_skip, raw_second_up], dim=1)  # 故意拼接空间尺寸不同的两路张量。
    naive_concat_failed = False  # 若意外成功则记录失败未被复现。
    naive_error_message = "未触发错误"  # 保存异常缺失说明。
except RuntimeError as error:  # 捕获 PyTorch 的真实尺寸不匹配异常。
    naive_concat_failed = True  # 标记失败案例已成功复现。
    naive_error_message = str(error).splitlines()[0]  # 保留首行错误信息供读者观察。
corrected_up = align_to(raw_second_up, target_skip)  # 用同一显式规则补齐到十七乘十九。
corrected_concat = torch.cat([target_skip, corrected_up], dim=1)  # 在对齐后安全沿通道维拼接。
print(f"错误行为：raw_up={tuple(raw_second_up.shape)}，skip={tuple(target_skip.shape)}，cat_error={naive_error_message}")  # 展示奇数尺寸导致的真实运行错误。
print(f"修正行为：aligned={tuple(corrected_up.shape)}，concat={tuple(corrected_concat.shape)}，空间尺寸完全一致。")  # 展示对称 padding 后的正确张量。

错误行为：raw_up=(8, 8, 16, 18)，skip=(8, 8, 17, 19)，cat_error=Sizes of tensors must match except in dimension 1. Expected size 17 but got size 16 for tensor number 1 in the list.
修正行为：aligned=(8, 8, 17, 19)，concat=(8, 16, 17, 19)，空间尺寸完全一致。


## 6. 生产边界

八张规则图会被网络记忆。生产需独立标注切分、标注者一致性与边界容差、正负像素采样、空 mask 策略、validation 选阈值、大图重叠 tile 和权重融合、相机域偏移、ONNX 转置卷积/resize 对齐，并监控逐类 IoU、边界 F1、假阳性面积和输入亮度漂移。

In [6]:
unet_diagnostics = {"images": len(images), "resolution": (height, width), "baseline_mean_iou": baseline_mean_iou, "unet_mean_iou": model_mean_iou, "initial_loss": history[0]["loss"], "final_loss": history[-1]["loss"], "raw_second_up": tuple(raw_second_up.shape), "aligned_second_up": tuple(corrected_up.shape), "naive_concat_failed": naive_concat_failed}  # 汇总数据、训练、像素和对齐指标。
print("生产监控快照：", unet_diagnostics)  # 输出分割系统应持续观察的信号。

生产监控快照： {'images': 8, 'resolution': (17, 19), 'baseline_mean_iou': 0.6424999833106995, 'unet_mean_iou': 1.0, 'initial_loss': 0.8047101497650146, 'final_loss': 2.760838469839655e-05, 'raw_second_up': (8, 8, 16, 18), 'aligned_second_up': (8, 8, 17, 19), 'naive_concat_failed': True}


## 7. 最小回归测试

最后一格只保护真实图像规模、训练更新、分割收益、输出尺寸和奇数对齐修正。

In [7]:
assert len(images) >= 6 and images.shape == masks.shape == (8, 1, 17, 19)  # 保证包含足够多奇数尺寸图像与真值。
assert history[-1]["loss"] < history[0]["loss"] and all(row["gradient_norm"] > 0.0 for row in history)  # 保证 U-Net 真实 backward 学习。
assert final_logits.shape == masks.shape and torch.isfinite(final_probabilities).all()  # 保证像素输出与输入空间严格一致且有限。
assert model_mean_iou > baseline_mean_iou and model_mean_iou >= 0.90  # 保证同数据 U-Net 明显优于固定亮度阈值。
assert naive_concat_failed and raw_second_up.shape[-2:] != target_skip.shape[-2:]  # 保证奇数尺寸直接 concat 的错误真实复现。
assert corrected_up.shape[-2:] == target_skip.shape[-2:] and corrected_concat.shape[1] == 16  # 保证显式对齐后空间与通道合同正确。